# 06 — Model Comparison (Multi-Model Pipeline)

Modular model registry with swappable classifiers. Compares LR, DT, RF,
LightGBM, CatBoost per playlist. Includes Isolation Forest anomaly scoring
and RFE feature selection comparison.

Historical JSONL metrics (from `train.py`) are loaded in sections 1-2.
Live training from `MODEL_REGISTRY` runs in section 3.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from paths import MODELS_DIR
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

RANDOM_STATE = 42
TEST_SIZE = 0.3

## 0. Model Registry & Evaluation

Swappable model registry — add or remove models with a single dict entry.
`evaluate_model()` standardizes train/predict/score across all classifiers.

In [ ]:
# --- Model Registry ---
# Add or remove models by editing this dict. Each entry needs an sklearn-compatible
# estimator and a type ("classifier" or "anomaly").
MODEL_REGISTRY: dict[str, dict] = {
    "logistic_regression": {
        "estimator": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
        ]),
        "type": "classifier",
    },
    "decision_tree": {
        "estimator": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        "type": "classifier",
    },
    "random_forest": {
        "estimator": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ]),
        "type": "classifier",
    },
    "lightgbm": {
        "estimator": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE, verbose=-1)),
        ]),
        "type": "classifier",
    },
    "catboost": {
        "estimator": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", CatBoostClassifier(iterations=100, random_seed=RANDOM_STATE, verbose=0)),
        ]),
        "type": "classifier",
    },
}

CLASSIFIER_NAMES = [k for k, v in MODEL_REGISTRY.items() if v["type"] == "classifier"]
LIVE_METRICS = ["accuracy", "f1", "precision", "recall", "roc_auc", "brier_score", "log_loss"]

# Direction: lower_better metrics for win/loss logic
LOWER_BETTER = {"brier_score", "log_loss"}


def evaluate_model(
    name: str,
    estimator: Pipeline,
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    """Train and evaluate a single classifier, returning standardized metrics."""
    est = clone(estimator)
    est.fit(X_train, y_train)
    y_pred = est.predict(X_test)

    result: dict = {"model": name, "accuracy": accuracy_score(y_test, y_pred)}

    # Probability-based metrics (if estimator supports predict_proba)
    if hasattr(est, "predict_proba"):
        y_prob = est.predict_proba(X_test)[:, 1]
        result["roc_auc"] = roc_auc_score(y_test, y_prob)
        result["brier_score"] = brier_score_loss(y_test, y_prob)
        result["log_loss"] = log_loss(y_test, y_prob)

    result["f1"] = f1_score(y_test, y_pred, zero_division=0)
    result["precision"] = precision_score(y_test, y_pred, zero_division=0)
    result["recall"] = recall_score(y_test, y_pred, zero_division=0)

    return result


print(f"Registry: {len(MODEL_REGISTRY)} models ({len(CLASSIFIER_NAMES)} classifiers)")
for name, entry in MODEL_REGISTRY.items():
    print(f"  {name:25s} [{entry['type']}]")

## 1. Load Metrics

In [ ]:
metrics_dir = MODELS_DIR / "metrics"
jsonl_files = sorted(metrics_dir.glob("*.jsonl")) if metrics_dir.exists() else []
print(f"Found {len(jsonl_files)} metrics files")

all_rows: list[dict] = []
summaries: list[dict] = []

for fpath in jsonl_files:
    for line in fpath.read_text().strip().split("\n"):
        if not line:
            continue
        entry = json.loads(line)
        entry["source_file"] = fpath.name
        if "summary" in entry:
            summaries.append(entry)
        else:
            # Flatten metrics into top-level
            metrics = entry.pop("metrics", {})
            entry.update(metrics)
            all_rows.append(entry)

if all_rows:
    df = pl.DataFrame(all_rows)
    print(f"Total metric entries: {len(df)}")
    print(f"Columns: {df.columns}")
    print(f"\nModes: {df['mode'].unique().to_list()}")
    print(f"Model types: {df['model_type'].unique().to_list()}")
    print(f"Playlists: {df['playlist'].n_unique()}")
    df.head(5)
else:
    print("No metrics data found — run train.py first")
    df = pl.DataFrame()

In [ ]:
# Show summaries
if summaries:
    print("Run summaries:")
    for s in summaries:
        print(f"  {s['source_file']}: {s.get('summary', {})}")

## 2. Filter to Compare Mode

Focus on `--compare` runs where both models were evaluated on the same playlists.

In [ ]:
if not df.is_empty():
    compare_df = df.filter(pl.col("mode") == "compare")
    print(f"Compare entries: {len(compare_df)}")

    if compare_df.is_empty():
        # Fall back to train mode if no compare runs exist
        print("No compare runs found — using train mode entries instead")
        compare_df = df

    # Use the most recent run (by timestamp)
    latest_ts = compare_df["timestamp"].unique().sort(descending=True)[0]
    latest = compare_df.filter(pl.col("timestamp") == latest_ts)
    print(f"Latest run: {latest_ts} ({len(latest)} entries)")
    latest.head()

## 2b. Live Multi-Model Training

Train all `MODEL_REGISTRY` classifiers per playlist using the corpus.
Binary label: 1 = track belongs to playlist, 0 = doesn't.

In [ ]:
# Load corpus and build per-playlist binary labels
conn = get_connection(read_only=True)
corpus = build_feature_matrix(conn)

AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]
available_feats = [f for f in AUDIO_FEATURES if f in corpus.columns]

# Get playlist membership
playlist_tracks = conn.execute("""
    SELECT pt.track_id, p.playlist_name
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
""").pl()
conn.close()

# Build feature matrix (numpy) — one row per track
X_all = corpus.select(available_feats).to_numpy()
track_ids = corpus["id"].to_list()
id_to_idx = {tid: i for i, tid in enumerate(track_ids)}

# Train all classifiers per playlist, collect results
live_rows: list[dict] = []
playlist_names = sorted(playlist_tracks["playlist_name"].unique().to_list())

for pname in playlist_names:
    # Binary label: 1 if track is in this playlist
    pos_ids = set(
        playlist_tracks.filter(pl.col("playlist_name") == pname)["track_id"].to_list()
    )
    y = np.array([1 if tid in pos_ids else 0 for tid in track_ids])

    # Skip if too few positives
    if y.sum() < 10:
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
    )

    for model_name in CLASSIFIER_NAMES:
        entry = MODEL_REGISTRY[model_name]
        result = evaluate_model(model_name, entry["estimator"], X_train, X_test, y_train, y_test)
        result["playlist"] = pname
        live_rows.append(result)

live_df = pl.DataFrame(live_rows)
print(f"Live training: {len(live_df)} results ({live_df['model'].n_unique()} models × {live_df['playlist'].n_unique()} playlists)")
live_df.head(10)

In [ ]:
# N-model grouped bar chart per metric (from live training)
if not live_df.is_empty():
    live_playlists = sorted(live_df["playlist"].unique().to_list())
    live_models = sorted(live_df["model"].unique().to_list())
    n_models = len(live_models)

    for metric in LIVE_METRICS:
        if metric not in live_df.columns:
            continue

        fig, ax = plt.subplots(figsize=(max(10, len(live_playlists) * 1.2), 5))
        x = np.arange(len(live_playlists))
        width = 0.8 / n_models

        for i, model_name in enumerate(live_models):
            model_data = live_df.filter(pl.col("model") == model_name).sort("playlist")
            vals = model_data[metric].to_numpy()
            offset = (i - n_models / 2 + 0.5) * width
            ax.bar(x + offset, vals, width, label=model_name, alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(live_playlists, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(metric)
        ax.set_title(f"{metric} — All Models (live)")
        ax.legend(fontsize=7)
        plt.tight_layout()
        plt.show()

In [ ]:
# Live mean metrics pivot: models × metrics (best highlighted)
if not live_df.is_empty():
    avail = [m for m in LIVE_METRICS if m in live_df.columns]
    mean_pivot = live_df.group_by("model").agg(
        [pl.col(m).mean().round(4).alias(m) for m in avail]
    ).sort("model")

    print("Mean metrics across playlists (live training):")
    print(mean_pivot)

    # Highlight best per metric
    print("\nBest model per metric:")
    for metric in avail:
        col_vals = mean_pivot[metric].to_list()
        model_names = mean_pivot["model"].to_list()
        if metric in LOWER_BETTER:
            best_idx = int(np.argmin(col_vals))
        else:
            best_idx = int(np.argmax(col_vals))
        print(f"  {metric:20s} → {model_names[best_idx]} ({col_vals[best_idx]:.4f})")

## 3. Per-Playlist Comparison

In [ ]:
KEY_METRICS = ["brier_score", "log_loss", "roc_auc", "precision_at_10", "f1"]

if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    lgbm = latest.filter(pl.col("model_type") == "lightgbm").sort("playlist")
    cat = latest.filter(pl.col("model_type") == "catboost").sort("playlist")

    # Ensure same playlists
    common = set(lgbm["playlist"].to_list()) & set(cat["playlist"].to_list())
    lgbm = lgbm.filter(pl.col("playlist").is_in(list(common))).sort("playlist")
    cat = cat.filter(pl.col("playlist").is_in(list(common))).sort("playlist")
    playlists = lgbm["playlist"].to_list()

    print(f"Comparing {len(playlists)} playlists")

    for metric in KEY_METRICS:
        if metric not in lgbm.columns:
            continue
        lgbm_vals = lgbm[metric].to_numpy()
        cat_vals = cat[metric].to_numpy()

        fig, ax = plt.subplots(figsize=(max(8, len(playlists) * 0.8), 5))
        x = np.arange(len(playlists))
        width = 0.35
        ax.bar(x - width/2, lgbm_vals, width, label="LightGBM", alpha=0.8)
        ax.bar(x + width/2, cat_vals, width, label="CatBoost", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(playlists, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(metric)
        ax.set_title(f"{metric} — LightGBM vs CatBoost")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Need both model types in the same run for comparison")

## 4. Win/Loss Summary

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    win_loss = []

    for metric in KEY_METRICS:
        if metric not in lgbm.columns:
            continue
        lgbm_vals = lgbm[metric].to_numpy()
        cat_vals = cat[metric].to_numpy()

        # For brier_score and log_loss, lower is better
        # For roc_auc, precision_at_10, f1, higher is better
        lower_better = metric in ("brier_score", "log_loss")

        if lower_better:
            lgbm_wins = int((lgbm_vals < cat_vals).sum())
            cat_wins = int((cat_vals < lgbm_vals).sum())
        else:
            lgbm_wins = int((lgbm_vals > cat_vals).sum())
            cat_wins = int((cat_vals > lgbm_vals).sum())

        ties = len(playlists) - lgbm_wins - cat_wins
        win_loss.append({
            "metric": metric,
            "lgbm_wins": lgbm_wins,
            "catboost_wins": cat_wins,
            "ties": ties,
            "lgbm_mean": round(float(lgbm_vals.mean()), 4),
            "catboost_mean": round(float(cat_vals.mean()), 4),
            "direction": "lower=better" if lower_better else "higher=better",
        })

    wl_df = pl.DataFrame(win_loss)
    print("Win/Loss Summary:")
    wl_df

## 5. Score Distributions (Box Plots)

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    fig, axes = plt.subplots(1, len(KEY_METRICS), figsize=(4 * len(KEY_METRICS), 5))
    if len(KEY_METRICS) == 1:
        axes = [axes]

    for i, metric in enumerate(KEY_METRICS):
        if metric not in latest.columns:
            continue
        data_pd = latest.select(["model_type", metric]).to_pandas()
        sns.boxplot(data=data_pd, x="model_type", y=metric, ax=axes[i])
        axes[i].set_title(metric)
        axes[i].set_xlabel("")

    plt.suptitle("Metric Distributions by Model Type", fontsize=13)
    plt.tight_layout()
    plt.show()

## 6. Metric Correlations

Does good Brier → good precision@10?

In [ ]:
if not latest.is_empty():
    available_metrics = [m for m in KEY_METRICS if m in latest.columns]

    for model_type in ["lightgbm", "catboost"]:
        model_df = latest.filter(pl.col("model_type") == model_type)
        if len(model_df) < 3:
            continue

        corr_data = model_df.select(available_metrics).to_pandas()
        corr = corr_data.corr()

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
        )
        ax.set_title(f"Metric Correlation — {model_type}")
        plt.tight_layout()
        plt.show()

## 7. Per-Playlist Deep Dive (Biggest Disagreements)

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    # Find playlists where models disagree most on Brier score
    if "brier_score" in lgbm.columns:
        brier_diff = np.abs(lgbm["brier_score"].to_numpy() - cat["brier_score"].to_numpy())
        top_diff_idx = np.argsort(brier_diff)[::-1][:5]

        print("Playlists with largest Brier score disagreement:")
        for idx in top_diff_idx:
            pname = playlists[idx]
            lgbm_row = lgbm.filter(pl.col("playlist") == pname)
            cat_row = cat.filter(pl.col("playlist") == pname)
            print(f"\n  {pname}:")
            for metric in KEY_METRICS:
                if metric in lgbm_row.columns:
                    lv = lgbm_row[metric][0]
                    cv = cat_row[metric][0]
                    delta = cv - lv
                    print(f"    {metric:20s}  LGBM={lv:.4f}  Cat={cv:.4f}  Δ={delta:+.4f}")

## 8. Temporal Stability (across runs)

In [ ]:
if not df.is_empty():
    timestamps = df["timestamp"].unique().sort().to_list()
    if len(timestamps) > 1:
        print(f"{len(timestamps)} distinct training runs found")

        # Mean Brier per run per model
        run_means = (
            df.filter(pl.col("brier_score").is_not_null())
            .group_by(["timestamp", "model_type"])
            .agg(pl.col("brier_score").mean().alias("mean_brier"))
            .sort("timestamp")
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        for mt in run_means["model_type"].unique().to_list():
            subset = run_means.filter(pl.col("model_type") == mt)
            ax.plot(subset["timestamp"].to_list(), subset["mean_brier"].to_list(), marker="o", label=mt)

        ax.set_xlabel("Run timestamp")
        ax.set_ylabel("Mean Brier Score")
        ax.set_title("Brier Score Stability Across Runs")
        ax.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Only 1 run found — need multiple runs for temporal comparison")

## 9. Isolation Forest Anomaly Scoring

Train IF on full corpus audio features. Surfaces stylistically unusual tracks
that don't fit any playlist's distribution well.

In [ ]:
# Isolation Forest on full corpus audio features
IF_N_ESTIMATORS = 150
IF_CONTAMINATION = 0.05

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_all)

iso = IsolationForest(
    n_estimators=IF_N_ESTIMATORS,
    contamination=IF_CONTAMINATION,
    random_state=RANDOM_STATE,
)
iso.fit(X_scaled)
iso_scores = iso.decision_function(X_scaled)

# Add scores to corpus for inspection
corpus_with_iso = corpus.with_columns(pl.Series("iso_score", iso_scores))

# Histogram of anomaly scores
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(iso_scores, bins=60, edgecolor="white", alpha=0.8)
ax.axvline(0, color="red", linestyle="--", label="anomaly threshold")
ax.set_xlabel("Isolation Forest decision score")
ax.set_title(f"Anomaly Score Distribution (contamination={IF_CONTAMINATION})")
ax.legend()
plt.tight_layout()
plt.show()

n_anomalies = int((iso_scores < 0).sum())
print(f"Anomalies (score < 0): {n_anomalies} / {len(iso_scores)} ({n_anomalies/len(iso_scores)*100:.1f}%)")

# Top-20 most anomalous tracks
display_cols = ["track_name", "iso_score"] + [f for f in available_feats if f in corpus.columns]
top_anomalies = (
    corpus_with_iso.select(display_cols)
    .sort("iso_score")
    .head(20)
)
print("\nTop 20 most anomalous tracks:")
top_anomalies

## 10. RFE Feature Selection Comparison

Recursive Feature Elimination on LR, DT, RF — which audio features does each
model family consider most important? Heatmap shows selected (1) vs dropped (0)
at 5 features retained.

In [ ]:
# RFE with 5 features retained across LR, DT, RF
RFE_N_FEATURES = 5

RFE_MODELS = {
    "logistic_regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

# Use the first playlist with enough positives as binary target for RFE
rfe_target = None
for pname in playlist_names:
    pos_ids = set(
        playlist_tracks.filter(pl.col("playlist_name") == pname)["track_id"].to_list()
    )
    y_rfe = np.array([1 if tid in pos_ids else 0 for tid in track_ids])
    if y_rfe.sum() >= 20:
        rfe_target = pname
        break

if rfe_target is not None:
    print(f"RFE target playlist: {rfe_target} ({int(y_rfe.sum())} positives)")

    # Scale features for RFE
    X_rfe = MinMaxScaler().fit_transform(X_all)

    selection_matrix = {}
    ranking_matrix = {}

    for model_name, estimator in RFE_MODELS.items():
        rfe = RFE(estimator=estimator, n_features_to_select=RFE_N_FEATURES, step=1)
        rfe.fit(X_rfe, y_rfe)
        selection_matrix[model_name] = rfe.support_.astype(int)
        ranking_matrix[model_name] = rfe.ranking_

    # Heatmap: features × models (1=selected, 0=dropped)
    sel_array = np.array([selection_matrix[m] for m in RFE_MODELS]).T
    rank_array = np.array([ranking_matrix[m] for m in RFE_MODELS]).T

    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(available_feats) * 0.5)))

    sns.heatmap(
        sel_array,
        xticklabels=list(RFE_MODELS.keys()),
        yticklabels=available_feats,
        annot=True, fmt="d", cmap="YlGn", ax=axes[0],
        cbar_kws={"label": "Selected (1) / Dropped (0)"},
    )
    axes[0].set_title(f"RFE Feature Selection (top {RFE_N_FEATURES})")

    sns.heatmap(
        rank_array,
        xticklabels=list(RFE_MODELS.keys()),
        yticklabels=available_feats,
        annot=True, fmt="d", cmap="YlOrRd_r", ax=axes[1],
        cbar_kws={"label": "Rank (1=best)"},
    )
    axes[1].set_title("RFE Feature Rankings")

    plt.tight_layout()
    plt.show()

    # Agreement: features selected by all 3 models
    all_selected = sel_array.sum(axis=1)
    unanimous = [f for f, s in zip(available_feats, all_selected) if s == len(RFE_MODELS)]
    partial = [f for f, s in zip(available_feats, all_selected) if 0 < s < len(RFE_MODELS)]
    never = [f for f, s in zip(available_feats, all_selected) if s == 0]
    print(f"\nUnanimously selected:  {unanimous}")
    print(f"Partially selected:    {partial}")
    print(f"Never selected:        {never}")
else:
    print("No playlist with enough positives for RFE")

## 10b. Stratified KFold Cross-Validation

5-fold CV per model to assess variance in metrics across folds.
Uses the same target playlist as RFE for consistency.

In [ ]:
# Stratified KFold cross-validation across all registry models
from sklearn.model_selection import StratifiedKFold, cross_validate

N_FOLDS = 5
CV_SCORING = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}

if rfe_target is not None:
    kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_results: list[dict] = []

    X_cv = MinMaxScaler().fit_transform(X_all)

    for model_name in CLASSIFIER_NAMES:
        estimator = clone(MODEL_REGISTRY[model_name]["estimator"])
        scores = cross_validate(
            estimator, X_cv, y_rfe,
            cv=kfold, scoring=CV_SCORING, return_train_score=False,
        )
        for metric_key, score_key in [("accuracy", "test_accuracy"), ("f1", "test_f1"), ("roc_auc", "test_roc_auc")]:
            vals = scores[score_key]
            cv_results.append({
                "model": model_name, "metric": metric_key,
                "mean": round(float(vals.mean()), 4),
                "std": round(float(vals.std()), 4),
                "min": round(float(vals.min()), 4),
                "max": round(float(vals.max()), 4),
            })

    cv_df = pl.DataFrame(cv_results)
    print(f"Stratified {N_FOLDS}-Fold CV — target: {rfe_target}")
    print(cv_df)

    # Box plot per metric
    fig, axes = plt.subplots(1, len(CV_SCORING), figsize=(5 * len(CV_SCORING), 5))
    for i, (metric_name, score_key) in enumerate(CV_SCORING.items()):
        fold_data = []
        for model_name in CLASSIFIER_NAMES:
            estimator = clone(MODEL_REGISTRY[model_name]["estimator"])
            scores = cross_validate(estimator, X_cv, y_rfe, cv=kfold, scoring=score_key)
            for val in scores["test_score"]:
                fold_data.append({"model": model_name, "score": val})
        fold_df = pl.DataFrame(fold_data).to_pandas()
        sns.boxplot(data=fold_df, x="model", y="score", ax=axes[i])
        axes[i].set_title(f"{metric_name} ({N_FOLDS}-fold)")
        axes[i].set_xlabel("")
        axes[i].tick_params(axis="x", rotation=30)

    plt.suptitle(f"Cross-Validation Stability — {rfe_target}", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No target playlist for CV")

## 10c. SMOTE Oversampling Comparison

Compare classifier performance with and without SMOTE on the minority class.
Imbalanced playlists (small positive set vs full corpus) benefit from synthetic
oversampling — but it can also hurt precision by introducing noisy positives.

In [ ]:
# SMOTE vs no-SMOTE comparison on LightGBM + Random Forest
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SMOTE_MODELS = {
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "lightgbm": LGBMClassifier(n_estimators=100, random_state=RANDOM_STATE, verbose=-1),
}

if rfe_target is not None:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X_all, y_rfe, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_rfe,
    )
    print(f"Target: {rfe_target} — train pos/neg: {y_train_s.sum()}/{(1-y_train_s).sum()}")

    smote_results: list[dict] = []
    for model_name, estimator in SMOTE_MODELS.items():
        # Without SMOTE
        pipe_no = Pipeline([("scaler", MinMaxScaler()), ("clf", clone(estimator))])
        pipe_no.fit(X_train_s, y_train_s)
        y_pred_no = pipe_no.predict(X_test_s)
        y_prob_no = pipe_no.predict_proba(X_test_s)[:, 1]

        # With SMOTE
        pipe_sm = ImbPipeline([
            ("scaler", MinMaxScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE)),
            ("clf", clone(estimator)),
        ])
        pipe_sm.fit(X_train_s, y_train_s)
        y_pred_sm = pipe_sm.predict(X_test_s)
        y_prob_sm = pipe_sm.predict_proba(X_test_s)[:, 1]

        for label, y_pred, y_prob in [("baseline", y_pred_no, y_prob_no), ("SMOTE", y_pred_sm, y_prob_sm)]:
            smote_results.append({
                "model": model_name, "sampling": label,
                "f1": round(f1_score(y_test_s, y_pred, zero_division=0), 4),
                "precision": round(precision_score(y_test_s, y_pred, zero_division=0), 4),
                "recall": round(recall_score(y_test_s, y_pred, zero_division=0), 4),
                "roc_auc": round(roc_auc_score(y_test_s, y_prob), 4),
                "brier": round(brier_score_loss(y_test_s, y_prob), 4),
            })

    smote_df = pl.DataFrame(smote_results)
    print("\nSMOTE comparison:")
    print(smote_df)

    # Grouped bar: baseline vs SMOTE per metric
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, metric in zip(axes, ["f1", "recall", "roc_auc"]):
        models = sorted(smote_df["model"].unique().to_list())
        x = np.arange(len(models))
        base_vals = [float(smote_df.filter((pl.col("model") == m) & (pl.col("sampling") == "baseline"))[metric][0]) for m in models]
        smote_vals = [float(smote_df.filter((pl.col("model") == m) & (pl.col("sampling") == "SMOTE"))[metric][0]) for m in models]
        ax.bar(x - 0.2, base_vals, 0.35, label="baseline", alpha=0.8)
        ax.bar(x + 0.2, smote_vals, 0.35, label="SMOTE", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=20)
        ax.set_title(metric)
        ax.legend(fontsize=8)
    plt.suptitle(f"SMOTE Impact — {rfe_target}", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No target playlist for SMOTE comparison")

## 10d. GridSearchCV Hyperparameter Tuning

Quick grid search on LightGBM and Decision Tree to show the effect of
key hyperparameters. Reports best params and CV score.

In [ ]:
# GridSearchCV on Decision Tree + LightGBM
from sklearn.model_selection import GridSearchCV
import pandas as pd

GRID_CONFIGS = {
    "decision_tree": {
        "estimator": Pipeline([("scaler", MinMaxScaler()), ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
        "param_grid": {
            "clf__max_depth": [2, 4, 6, 8, None],
            "clf__criterion": ["gini", "entropy"],
            "clf__splitter": ["best", "random"],
        },
    },
    "lightgbm": {
        "estimator": Pipeline([("scaler", MinMaxScaler()), ("clf", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))]),
        "param_grid": {
            "clf__n_estimators": [50, 100, 200],
            "clf__max_depth": [3, 5, 7, -1],
            "clf__learning_rate": [0.01, 0.1, 0.2],
        },
    },
}

if rfe_target is not None:
    kfold_gs = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    grid_results: list[dict] = []

    for model_name, config in GRID_CONFIGS.items():
        gs = GridSearchCV(
            config["estimator"], config["param_grid"],
            cv=kfold_gs, scoring="roc_auc", n_jobs=-1, refit=True,
        )
        gs.fit(X_all, y_rfe)

        best_params = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}
        grid_results.append({
            "model": model_name,
            "best_roc_auc": round(gs.best_score_, 4),
            "best_params": str(best_params),
        })
        print(f"{model_name}: best ROC-AUC={gs.best_score_:.4f}")
        print(f"  params: {best_params}")

    grid_df = pl.DataFrame(grid_results)
    print("\nGridSearchCV Summary:")
    print(grid_df)

    # Heatmap: DT max_depth × criterion (use pandas for cv_results — mixed types)
    dt_gs = GridSearchCV(
        GRID_CONFIGS["decision_tree"]["estimator"],
        GRID_CONFIGS["decision_tree"]["param_grid"],
        cv=kfold_gs, scoring="roc_auc", n_jobs=-1,
    )
    dt_gs.fit(X_all, y_rfe)
    cv_res_pd = pd.DataFrame(dt_gs.cv_results_)

    # Replace None with "None" for display, filter to splitter="best"
    cv_best = cv_res_pd[cv_res_pd["param_clf__splitter"] == "best"].copy()
    cv_best["depth_label"] = cv_best["param_clf__max_depth"].apply(lambda x: str(x) if x is not None else "None")

    depths_labels = sorted(cv_best["depth_label"].unique(), key=lambda x: (x == "None", int(x) if x != "None" else 999))
    criteria = sorted(cv_best["param_clf__criterion"].unique())

    heatmap_data = np.zeros((len(depths_labels), len(criteria)))
    for i, dl in enumerate(depths_labels):
        for j, c in enumerate(criteria):
            row = cv_best[(cv_best["depth_label"] == dl) & (cv_best["param_clf__criterion"] == c)]
            if not row.empty:
                heatmap_data[i, j] = float(row["mean_test_score"].iloc[0])

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        heatmap_data,
        xticklabels=criteria,
        yticklabels=depths_labels,
        annot=True, fmt=".3f", cmap="YlGn", ax=ax,
    )
    ax.set_xlabel("Criterion")
    ax.set_ylabel("Max Depth")
    ax.set_title("Decision Tree: ROC-AUC by max_depth × criterion")
    plt.tight_layout()
    plt.show()
else:
    print("No target playlist for GridSearch")

## 11. Recommendation Summary

In [ ]:
if not latest.is_empty() and latest["model_type"].n_unique() >= 2:
    print("=" * 60)
    print("MODEL COMPARISON SUMMARY")
    print("=" * 60)
    print(f"Playlists compared: {len(playlists)}")
    print(f"Run: {latest_ts}")
    print()

    for _, row in wl_df.iter_rows(named=True):
        pass  # just need last

    for row in wl_df.iter_rows(named=True):
        winner = "LightGBM" if row["lgbm_wins"] > row["catboost_wins"] else "CatBoost" if row["catboost_wins"] > row["lgbm_wins"] else "Tie"
        print(f"{row['metric']:20s}  Winner: {winner:10s}  (LGBM {row['lgbm_mean']:.4f} vs Cat {row['catboost_mean']:.4f})")

    print()
    print("Decision criteria:")
    print("  - Brier score (primary): lower = better calibrated probabilities for reranking")
    print("  - Log-loss: lower = better probability estimates")
    print("  - Precision@10: higher = better top-k rerank quality")
    print("  - ROC-AUC: higher = better discrimination")
else:
    print("Insufficient data for summary")